# Structural z-stack soma/process segmentation
Average repeated ScanImage frames within each plane, curate soma profiles and processes plane-by-plane, and export depth-resolved tables. There is no odor or trace alignment in this workflow.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

# ---- settings to verify for each stack ----
TIFF = Path('/Volumes/MossLab/ImagingData/20260710/m357/zstack2/20260710_m357_zstack2_00001.tif')
FRAMES_PER_PLANE = None  # None reads ScanImage metadata (10 in the test file)
CHANNEL = 0              # zero-based; test file contains one saved channel
UM_PER_PX = 1.0          # calibrated value; this TIFF does not store a micron-scale calibration
MIN_SOMA_DIAMETER_UM = 10  # omit smaller soma profiles from exported tables and statistics
CENTER_DEPTH_UM = -150   # depth of the center plane relative to surface; set None to use DEPTH_ZERO_PLANE
DEPTH_ZERO_PLANE = 0     # alternative: zero-based plane assigned depth 0 when CENTER_DEPTH_UM is None
DEPTH_DIRECTION = 1      # +1: depth becomes numerically larger with plane index; -1: more negative
LINE = 'thy1'            # change to 'th' or 'dat'; stored below in metadata
OUTPUT = TIFF.with_name(TIFF.stem + '_structural_segmentation.h5')
# -------------------------------------------

default_root = Path.cwd() if (Path.cwd() / 'analysis').is_dir() else Path.cwd().parent
analysis_root = Path(os.environ.get('ODYN_ANALYSIS_ROOT', default_root)).resolve()
if str(analysis_root) not in sys.path: sys.path.insert(0, str(analysis_root))
from analysis.seg_zstack import StructuralZStackState, load_scanimage_zstack
from analysis.seg_zstack.gui import launch
print('input:', TIFF)
print('output:', OUTPUT)
if UM_PER_PX is None:
    print('WARNING: UM_PER_PX is unset; outputs will remain pixel-based and physical density will be omitted.')

## Load and average the repeated images per plane

In [ ]:
if OUTPUT.exists():
    state = StructuralZStackState.load(OUTPUT)
    print(f'Resuming {len(state.planes)} planes in phase {state.phase!r}')
else:
    structural, metadata = load_scanimage_zstack(
        TIFF, frames_per_plane=FRAMES_PER_PLANE, channel=CHANNEL, progress=True,
    )
    metadata['line'] = LINE
    state = StructuralZStackState(structural, metadata=metadata)
    print(metadata)
    print('averaged stack:', structural.shape, structural.dtype)

## Curate in the plane browser
Parameters apply to every plane. Tune on several representative depths, then click the blue advance button to automatically segment all planes. During curation, use the plane slider and click masks to delete or background to add. Cyan is soma; magenta is process. Save writes the HDF5 bundle plus ROI-level and depth-summary CSV files.

In [ ]:
gui = launch(
    state, save_path=OUTPUT, um_per_px=UM_PER_PX,
    min_soma_diameter_um=MIN_SOMA_DIAMETER_UM,
    depth_zero_plane=DEPTH_ZERO_PLANE, center_depth_um=CENTER_DEPTH_UM,
    depth_direction=DEPTH_DIRECTION,
)

## Inspect depth results after saving

In [ ]:
import pandas as pd
summary_path = OUTPUT.with_name(OUTPUT.stem + '_depth_summary.csv')
roi_path = OUTPUT.with_name(OUTPUT.stem + '_rois.csv')
summary = pd.read_csv(summary_path)
rois = pd.read_csv(roi_path)
display(summary)
display(rois[rois.roi_type == 'soma'].head())

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
has_depth = summary.depth_um.notna().any()
x_col, x_label = ('depth_um', 'depth relative to surface (µm)') if has_depth else ('plane', 'plane (0 based)')
axes[0].plot(summary[x_col], summary.n_soma_profiles, 'o-', color='deepskyblue')
axes[0].set(xlabel=x_label, ylabel=f'soma profiles ≥{MIN_SOMA_DIAMETER_UM:g} µm', title=f'{LINE}: soma count by depth')
somas = rois[rois.roi_type == 'soma']
size_col, size_label = ('area_um2', 'soma profile area (µm²)') if 'area_um2' in somas else ('area_px', 'soma profile area (pixels)')
positions = summary[x_col].to_numpy()
box_data = [somas.loc[somas.plane == plane, size_col].dropna().to_numpy() for plane in summary.plane]
keep = [i for i, values in enumerate(box_data) if len(values)]
if keep:
    spacing = max(1.0, float(abs(positions[1] - positions[0]))) if len(positions) > 1 else 1.0
    axes[1].boxplot([box_data[i] for i in keep], positions=positions[keep], widths=.65*spacing,
                    patch_artist=True, boxprops={'facecolor':'deepskyblue', 'alpha':.55},
                    medianprops={'color':'black'})
axes[1].set(xlabel=x_label, ylabel=size_label, title=f'{LINE}: soma size by depth')
for ax in axes: ax.grid(alpha=.2)
plt.show()

### Interpretation note
Counts are **2D soma profiles per optical plane**, not stereologically corrected 3D cell counts. A soma visible in adjacent planes can therefore contribute more than once. Use these tables consistently across TH and DAT lines for profile-density comparisons, or add cross-plane object linking/stereology before interpreting them as volumetric cell density.